# **EVALUASI PEMAHAMAN BAHASA LISAN (NLU): AKURASI, LOSS, DAN F1-SCORE**

Notebook ini menjawab dua rumusan masalah sekaligus:

- **RM-4 — perancangan.** Bagaimana merancang mekanisme pemahaman makna
  (*Natural Language Understanding*) berbasis aturan untuk mengekstraksi *intent*
  transaksi serta entitas **produk**, **jumlah**, dan **harga** dari teks hasil
  transkripsi perintah suara pedagang. Rancangannya dibangun pada bagian
  *Rancangan NLU berbasis aturan* dan dapat dijalankan langsung di notebook ini.
- **RM-5 — pengukuran.** Bagaimana performansi model pemahaman bahasa lisan
  diukur dengan **akurasi** dan **F1-Score** dalam mengklasifikasikan *intent*
  dan mengekstraksi entitas dari teks hasil transkripsi.

**Tiga kondisi masukan dievaluasi**, dan perbedaannya penting:

| kondisi | masukan NLU | yang terukur |
|---|---|---|
| teks acuan | transkrip manusia | mutu aturan NLU itu sendiri (batas atas) |
| teks ASR *speaker-dependent* | keluaran model pada penutur yang dikenal | kinerja ujung-ke-ujung, kondisi menguntungkan |
| teks ASR *speaker-independent* | keluaran model pada penutur baru | kinerja ujung-ke-ujung, kondisi lapangan |

Selisih antara kondisi pertama dan ketiga adalah **biaya kesalahan transkripsi**
terhadap pemahaman makna — informasi yang tidak dapat dilihat dari CER/WER saja.

**Catatan tentang "loss".** NLU di sini berbasis aturan, bukan model yang
dilatih, sehingga tidak memiliki fungsi loss yang dioptimalkan dan tidak
menghasilkan probabilitas kelas — *cross-entropy* tidak terdefinisi untuknya.
Yang dilaporkan adalah dua ukuran loss yang memang berlaku untuk pengklasifikasi
deterministik, yaitu **0-1 loss** dan **Hamming loss**, ditambah **kurva loss
pelatihan model CTC** bila riwayat pelatihannya tersedia. Alasannya dijelaskan di
bagian *Loss*.

**Prasyarat:** jalankan `07_evaluasi_cer_wer.ipynb` lebih dulu — notebook itu
menghasilkan `transkripsi_<skenario>.csv` yang dipakai di sini. Tanpa berkas
tersebut, notebook ini tetap berjalan tetapi hanya untuk kondisi teks acuan.

# **IMPORT LIBRARY**

Tahap ini memuat pustaka untuk pengolahan teks, perhitungan metrik klasifikasi,
serta visualisasi hasil.

In [ ]:
import os, re, csv, json, glob, time, unicodedata, collections
from difflib import SequenceMatcher

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support, classification_report,
    confusion_matrix,
)

plt.rcParams['figure.dpi'] = 110
plt.rcParams['savefig.bbox'] = 'tight'

import sklearn
print('pandas      :', pd.__version__)
print('scikit-learn:', sklearn.__version__)

**Insight:**

Metrik klasifikasi dihitung memakai `scikit-learn`, bukan implementasi sendiri.
Definisi *precision*, *recall*, dan F1 — terutama perbedaan rata-rata *macro*
dan *weighted* pada kelas yang tidak seimbang — mudah keliru bila ditulis ulang,
dan kekeliruannya menghasilkan angka yang tetap terlihat masuk akal.

# **KONFIGURASI**

Tahap ini menetapkan lokasi transkrip acuan, katalog produk beserta harga
satuannya, hasil transkripsi dari notebook 07, serta folder keluaran.

Katalog produk (`unit_price_lookup.json`) adalah berkas yang sama dengan yang
dipakai layanan REST FinSense. Memakai berkas yang sama memastikan daftar produk
pada evaluasi ini identik dengan yang dikenali aplikasi.

In [ ]:
KANDIDAT_DATASET = [
    '/kaggle/input/finsense-updated-dataset',
    '/kaggle/input/datasets/ibrahimirfanulhaq/finsense-updated-dataset',
    os.path.expanduser('~/finsense_expanded'),
    '../data/raw/finsense_expanded',
]


def cari(pola_list):
    for pola in pola_list:
        hit = sorted(glob.glob(pola, recursive=True))
        if hit:
            return hit[0]
    return None


TRANSCRIPTS = cari([os.path.join(a, '**', 'transcripts_expanded.csv') for a in KANDIDAT_DATASET]
                   + ['/kaggle/input/**/transcripts_expanded.csv'])
# Katalog produk + harga satuan. Tidak wajib: bila tidak ada, daftar produk
# dibentuk dari korpus Tahap 1 dan bagian yang memang butuh harga akan
# dilewati dengan pemberitahuan (lihat sel LOAD DATA).
KATALOG = cari([
    '../../ai-service/model_v10/unit_price_lookup.json',
    '/kaggle/input/**/unit_price_lookup.json',
    './unit_price_lookup.json',
])
assert TRANSCRIPTS, 'transcripts_expanded.csv tidak ditemukan.'

# Hasil notebook 07 (boleh belum ada; kondisi ASR akan dilewati bila kosong).
#
# Tiga tempat dicari, berurutan:
#   1. /kaggle/working  — notebook 07 dijalankan di sesi yang sama
#   2. ../hasil/...     — dijalankan lokal
#   3. /kaggle/input    — keluaran notebook 07 disimpan sebagai dataset lalu
#                         di-attach ke sini. Ini yang berlaku bila 07 dan 08
#                         dijalankan sebagai dua notebook Kaggle terpisah,
#                         sebab isi /kaggle/working hilang saat sesi berakhir.
DIR_ASR = ('/kaggle/working/hasil_evaluasi_asr' if os.path.isdir('/kaggle/working/hasil_evaluasi_asr')
           else '../hasil/evaluasi_asr')


def cari_berkas_asr(nama_berkas):
    lokal = os.path.join(DIR_ASR, nama_berkas)
    if os.path.exists(lokal):
        return lokal
    hit = sorted(glob.glob(f'/kaggle/input/**/{nama_berkas}', recursive=True))
    return hit[0] if hit else lokal   # kembalikan jalur lokal agar pesannya jelas


BERKAS_ASR = {
    'asr_speaker_dependent': cari_berkas_asr('transkripsi_speaker_dependent.csv'),
    'asr_speaker_independent': cari_berkas_asr('transkripsi_speaker_independent.csv'),
    # Teks hasil model yang dilayankan (v12) atas klip validasi korpus Tahap 1.
    'asr_tahap1_val': cari_berkas_asr('transkripsi_tahap1_val.csv'),
}

# Korpus Tahap 1: manifest klip potongan (produk saja, intent+produk, produk+qty,
# kalimat penuh). Dipakai sebagai kondisi masukan tambahan agar NLU juga diuji
# pada ucapan sepotong, bukan hanya kalimat penuh.
def cari_manifest_tahap1():
    kandidat = [
        os.path.expanduser('~/finsense_tahap1/manifest_val.csv'),
        '../../korpus_tahap1/manifest_val.csv',
        './manifest_val.csv',
    ]
    for f in kandidat:
        if f and os.path.exists(f):
            return f
    hit = sorted(glob.glob('/kaggle/input/**/manifest_val.csv', recursive=True))
    return hit[0] if hit else None


MANIFEST_TAHAP1 = cari_manifest_tahap1()

KELUARAN = ('/kaggle/working/hasil_evaluasi_nlu' if os.path.isdir('/kaggle/working')
            else '../hasil/evaluasi_nlu')
os.makedirs(KELUARAN, exist_ok=True)

# Seed untuk bagian yang memakai keacakan (pemilihan produk yang
# disembunyikan pada uji penolakan). Ditetapkan agar hasilnya sama setiap
# kali notebook dijalankan.
SEED = 42

# Ambang kemiripan minimum agar sebuah span teks diterima sebagai nama produk.
# Di bawah ini, NLU menjawab "tidak dikenali" alih-alih menebak. Nilainya
# dikalibrasi pada bagian Kalibrasi Ambang di notebook ini.
AMBANG_PRODUK = 0.45

print('transkrip acuan :', TRANSCRIPTS)
print('katalog produk  :', KATALOG or 'tidak ada — daftar produk akan diambil dari korpus Tahap 1')
print('keluaran        :', os.path.abspath(KELUARAN))
print('manifest Tahap 1:', MANIFEST_TAHAP1 or 'tidak ada — kondisi Tahap 1 dilewati')
for nama, path in BERKAS_ASR.items():
    print(f'{nama:24s}:', path if os.path.exists(path) else 'BELUM ADA (jalankan notebook 07)')

**Insight:**

Ambang `AMBANG_PRODUK` adalah satu-satunya bilangan bebas pada rancangan NLU
ini. Ia menentukan kapan sistem memilih menjawab "tidak dikenali" daripada
menebak produk yang salah. Untuk pencatatan transaksi, menolak jauh lebih murah
daripada salah: transaksi yang tidak tercatat akan segera disadari kasir,
sedangkan transaksi yang tercatat dengan produk keliru baru ketahuan saat stok
tidak cocok. Nilainya dikalibrasi, bukan ditebak — lihat bagian kalibrasi.

# **LOAD DATA**

Tahap ini memuat transkrip acuan dan katalog produk, lalu menormalkan teksnya
dengan aturan yang sama seperti tahap pelatihan model (huruf kecil, hanya `a-z`
dan spasi) agar keduanya berada pada ruang simbol yang sama.

In [ ]:
def bersihkan_teks(t):
    """Normalisasi identik dengan tahap pelatihan: huruf kecil, hanya a-z dan spasi."""
    t = unicodedata.normalize('NFKD', str(t)).lower().strip()
    t = re.sub(r'[^a-z ]', '', t)
    return re.sub(r'\s+', ' ', t)


df = pd.read_csv(TRANSCRIPTS)
df['teks'] = df['transcript'].apply(bersihkan_teks)

if KATALOG:
    with open(KATALOG, encoding='utf-8') as f:
        art = json.load(f)
    PRODUK_VOCAB = [bersihkan_teks(p) for p in art['produk_vocab']]
    HARGA_SATUAN = {bersihkan_teks(k): int(v) for k, v in art['unit_price_lookup'].items()}
elif MANIFEST_TAHAP1:
    # Cadangan: korpus Tahap 1 memuat varian 'produk' yang isinya persis nama
    # produk, jadi daftar namanya dapat dibaca langsung dari manifest. Harga
    # satuan TIDAK ada di sana, dan tidak boleh dikarang — bagian yang
    # membutuhkannya dilewati dan dinyatakan di keluaran.
    # manifest.csv (korpus penuh) dipakai bila ada, bukan manifest_val.csv:
    # manifest validasi hanya memuat sebagian produk, dan katalog yang tidak
    # lengkap membuat produk yang hilang MUSTAHIL dikenali — angkanya turun
    # bukan karena NLU-nya buruk, melainkan karena katalognya bolong.
    _man_penuh = os.path.join(os.path.dirname(MANIFEST_TAHAP1), 'manifest.csv')
    _sumber_katalog = _man_penuh if os.path.exists(_man_penuh) else MANIFEST_TAHAP1
    _man = pd.read_csv(_sumber_katalog)
    PRODUK_VOCAB = sorted({bersihkan_teks(t) for t, v in
                           zip(_man['transcript'], _man['variant']) if v == 'produk'})
    HARGA_SATUAN = {}
    print('PERHATIAN: unit_price_lookup.json tidak ditemukan.')
    print(f'  Daftar produk diambil dari {os.path.basename(_sumber_katalog)} '
          f'(varian "produk"): {len(PRODUK_VOCAB)} nama.')
    print('  Harga satuan tidak tersedia, sehingga yang DILEWATI adalah:')
    print('    - pemeriksaan silang harga = harga_satuan x jumlah')
    print('    - kolom harga_dihitung pada keluaran NLU')
    print('  Evaluasi intent dan entitas tetap berjalan penuh.')
    print('  Unggah ai-service/model_v10/unit_price_lookup.json (9 KB) untuk melengkapinya.')
    if len(PRODUK_VOCAB) < 121:
        print(f'  CATATAN: katalog hasil cadangan berisi {len(PRODUK_VOCAB)} produk, '
              'sedangkan aplikasi mengenali 121.')
        print('  Produk yang tidak ada di daftar ini mustahil dikenali, sehingga '
              'angka di bawah lebih rendah daripada kemampuan sebenarnya.')
    print()
else:
    raise FileNotFoundError(
        'Katalog produk tidak ditemukan dan korpus Tahap 1 juga tidak ada. '
        'Sediakan salah satu: unit_price_lookup.json, atau manifest korpus Tahap 1.')

# Pencocokan persis memakai himpunan nama, bukan tabel harga, supaya tetap
# bekerja saat harga tidak tersedia.
PRODUK_SET = set(PRODUK_VOCAB)

# Diurutkan dari nama terpanjang: penjangkaran harus mencoba "abc kecap asin
# enam ratus dua puluh mililiter" sebelum "kecap asin enam ratus dua puluh
# mililiter", agar tidak berhenti pada potongan nama yang lebih pendek.
PRODUK_URUT = sorted(set(PRODUK_VOCAB), key=lambda p: -len(p.split()))

print(f'ucapan        : {len(df):,}')
print(f'produk katalog: {len(PRODUK_URUT)}')
print(f'harga satuan  : {len(HARGA_SATUAN)} entri' if HARGA_SATUAN else 'harga satuan  : tidak tersedia')
print(f'contoh produk : {PRODUK_URUT[0]}')
df[['filename', 'teks']].head()

**Insight:**

Katalog diurutkan menurun berdasarkan jumlah kata. Urutan ini menentukan
kebenaran penjangkaran: beberapa nama produk merupakan awalan atau akhiran dari
nama lain (`kecap asin ...` ada di dalam `abc kecap asin ...`). Tanpa
pengurutan, penjangkaran dapat berhenti pada nama yang lebih pendek dan
menghasilkan entitas produk yang salah, padahal teksnya benar.

# **PELABELAN ACUAN (GOLD LABEL)**

Evaluasi apa pun hanya sebaik label acuannya. Karena itu label acuan di sini
**tidak** dibuat oleh parser yang sedang dinilai — kalau begitu caranya, akurasi
akan 100% dengan sendirinya dan tidak berarti apa-apa.

Label acuan dibangun dari struktur cara dataset ini dibentuk:

1. **Produk** — penjangkaran kosakata: mencari rentang kata pada transkrip yang
   **persis sama** dengan salah satu nama di katalog. Bukan pencocokan samar,
   bukan aturan pola kalimat.
2. **Intent** — kata pembuka ucapan bila termasuk daftar kata intent; bila tidak
   ada, labelnya `tanpa_penanda`.
3. **Jumlah** — kata bilangan pertama di luar rentang nama produk.
4. **Harga** — deretan kata bilangan tepat sebelum kata `rupiah`, bila
   disebutkan.

Yang membuat label ini dapat dipercaya adalah **pemeriksaan silang**: pada
ucapan yang menyebut harga, nilai `harga` hasil pelabelan harus sama dengan
`harga satuan katalog x jumlah`. Ketiga label (produk, jumlah, harga) berasal
dari sumber yang berbeda, sehingga kesesuaiannya tidak mungkin kebetulan. Hasil
pemeriksaan ditampilkan di bawah.

In [ ]:
KATA_INTENT = {'jual', 'laku', 'keluar', 'tolong', 'beli', 'tambah', 'jualan', 'pesan'}
INTENT_KANONIK = {
    'jual': 'catat_penjualan', 'laku': 'catat_penjualan',
    'keluar': 'catat_penjualan', 'tolong': 'catat_penjualan',
    'jualan': 'catat_penjualan', 'pesan': 'catat_penjualan',
    'beli': 'catat_pembelian', 'tambah': 'catat_pembelian',
    'tanpa_penanda': 'catat_penjualan',   # bawaan aplikasi: POS mencatat penjualan
}

BILANGAN = {'satu': 1, 'dua': 2, 'tiga': 3, 'empat': 4, 'lima': 5,
            'enam': 6, 'tujuh': 7, 'delapan': 8, 'sembilan': 9, 'sepuluh': 10}
SKALA = {'ratus', 'puluh', 'belas', 'ribu', 'juta', 'seratus', 'seribu', 'sebelas'}
KATA_ANGKA = set(BILANGAN) | SKALA

# Satuan HITUNG: menyatakan banyaknya barang.
SATUAN_HITUNG = {'bungkus', 'botol', 'dus', 'pcs', 'pack', 'pak', 'renteng', 'slop',
                 'sak', 'biji', 'buah', 'lusin', 'karton', 'box', 'kaleng',
                 'sachet', 'tray', 'ikat', 'lembar', 'batang', 'butir'}
# Satuan UKURAN: menerangkan isi kemasan, bagian dari NAMA produk.
# "aqua enam ratus mililiter" -> 'enam' di sini bukan jumlah barang.
SATUAN_UKURAN = {'mililiter', 'militer', 'liter', 'gram', 'kilogram', 'kilo', 'ml', 'kg', 'gr'}


def teks_ke_angka(kata_kata):
    """Deretan kata bilangan bahasa Indonesia -> bilangan bulat.

    'dua puluh empat ribu' -> 24000 ; 'seratus delapan puluh' -> 180
    """
    def ratusan(toks):
        nilai = 0
        for i, w in enumerate(toks):
            if w == 'seratus':
                nilai += 100
            elif w == 'ratus':
                sebelum = BILANGAN.get(toks[i - 1], 0) if i else 0
                nilai = nilai - sebelum + max(sebelum, 1) * 100
            elif w == 'sepuluh':
                nilai += 10
            elif w == 'sebelas':
                nilai += 11
            elif w == 'belas':
                sebelum = BILANGAN.get(toks[i - 1], 0) if i else 0
                nilai = nilai - sebelum + (10 + sebelum)
            elif w == 'puluh':
                sebelum = BILANGAN.get(toks[i - 1], 0) if i else 0
                nilai = nilai - sebelum + sebelum * 10
            elif w in BILANGAN:
                nilai += BILANGAN[w]
        return nilai

    kata_kata = [w for w in kata_kata if w in KATA_ANGKA]
    if not kata_kata:
        return None
    if 'ribu' in kata_kata:
        i = kata_kata.index('ribu')
        depan = kata_kata[:i]
        return (ratusan(depan) if depan else 1) * 1000 + ratusan(kata_kata[i + 1:])
    if 'seribu' in kata_kata:
        return 1000 + ratusan(kata_kata[kata_kata.index('seribu') + 1:])
    return ratusan(kata_kata)


def jangkar_produk(kata_kata):
    """Rentang kata yang persis sama dengan sebuah nama di katalog -> (nama, awal, akhir)."""
    for nama in PRODUK_URUT:
        pola = nama.split()
        n = len(pola)
        for i in range(len(kata_kata) - n + 1):
            if kata_kata[i:i + n] == pola:
                return nama, i, i + n
    return None, -1, -1


def label_acuan(teks):
    """Label acuan satu ucapan: intent, produk, jumlah, harga."""
    w = teks.split()
    produk, awal, akhir = jangkar_produk(w)
    intent = w[0] if w and w[0] in KATA_INTENT else 'tanpa_penanda'

    luar = w[:awal] + w[akhir:] if produk else list(w)

    # Jumlah: kata bilangan pertama di luar nama produk. Kata bilangan yang
    # muncul kemudian adalah bagian nominal harga ("dua puluh empat ribu").
    jumlah = next((BILANGAN[x] for x in luar if x in BILANGAN), None)

    # Harga: deretan kata bilangan yang bersambung tepat sebelum 'rupiah'.
    #
    # Penanda cadangan: pada klip korpus Tahap 1, kata 'rupiah' kerap terpangkas
    # oleh batas jendela 8 detik sementara nominalnya masih utuh terucap
    # ("... satu bungkus delapan belas ribu"). Deretan bilangan di ujung ucapan
    # yang berakhir 'ribu'/'juta' karena itu ikut diterima sebagai nominal.
    # Pada transkrip finsense_expanded aturan cadangan ini tidak pernah aktif:
    # di sana tidak ada satu pun ucapan yang berakhir 'ribu'/'juta' tanpa
    # 'rupiah', jadi angka kondisi teks acuan tidak berubah karenanya.
    harga = None
    if 'rupiah' in w:
        i = len(w) - 1 - w[::-1].index('rupiah')
        j = i - 1
        while j >= 0 and w[j] in KATA_ANGKA:
            j -= 1
        if j + 1 < i:
            harga = teks_ke_angka(w[j + 1:i])
    elif w and w[-1] in ('ribu', 'juta'):
        j = len(w) - 1
        while j >= 0 and w[j] in KATA_ANGKA:
            j -= 1
        if j + 1 < len(w):
            harga = teks_ke_angka(w[j + 1:])

    return {'intent': intent, 'intent_kanonik': INTENT_KANONIK.get(intent, 'catat_penjualan'),
            'produk': produk, 'jumlah': jumlah, 'harga': harga}


acuan = pd.DataFrame([label_acuan(t) for t in df['teks']])
acuan.insert(0, 'filename', df['filename'].values)
acuan.insert(1, 'teks', df['teks'].values)

# ─── Pemeriksaan silang: harga = harga satuan katalog x jumlah ───
punya_harga = (acuan['harga'].notna() & acuan['jumlah'].notna()
               & acuan['produk'].notna() & bool(HARGA_SATUAN))
cek = acuan[punya_harga].copy()
cek['harga_hitung'] = cek.apply(lambda r: HARGA_SATUAN.get(r['produk'], -1) * r['jumlah'], axis=1)
cocok = int((cek['harga'] == cek['harga_hitung']).sum())

print(f'produk terjangkar : {acuan.produk.notna().sum():,}/{len(acuan):,} '
      f'({100*acuan.produk.notna().mean():.1f}%)')
print(f'jumlah terlabeli  : {acuan.jumlah.notna().sum():,}/{len(acuan):,}')
print(f'harga disebutkan  : {acuan.harga.notna().sum():,}/{len(acuan):,}')
print(f'\nPemeriksaan silang harga = harga_satuan x jumlah:')
if not HARGA_SATUAN:
    print('  DILEWATI — tabel harga satuan tidak tersedia.')
else:
    print(f'  cocok: {cocok:,}/{len(cek):,} ({100*cocok/max(len(cek),1):.2f}%)')
if cocok < len(cek):
    print('\n  Contoh yang tidak cocok (perlu diperiksa sebelum angka dipakai):')
    print(cek[cek.harga != cek.harga_hitung].head(5)[['teks', 'produk', 'jumlah', 'harga', 'harga_hitung']].to_string(index=False))

print('\nSebaran intent acuan:')
print(acuan.intent.value_counts().to_string())
acuan.head()

**Insight:**

Pemeriksaan silang harga adalah pengaman utama tahap ini. Label produk berasal
dari penjangkaran kosakata, label jumlah dari posisi kata bilangan, dan label
harga dari nominal yang diucapkan — tiga jalur yang saling bebas. Bila
ketiganya konsisten pada seluruh ucapan yang menyebut harga, kemungkinan besar
ketiganya benar; bila tidak, yang salah harus diperbaiki **sebelum** metrik apa
pun dihitung, sebab metrik yang dibandingkan dengan acuan yang salah hanya
mengukur kesalahan acuannya.

Perhatikan juga kelas `tanpa_penanda` pada sebaran intent. Kelas ini muncul
karena sebagian pedagang langsung menyebut produk tanpa kata kerja pembuka
("indomie goreng dua"). Aplikasi tetap harus menangani ucapan seperti itu, jadi
ia diperlakukan sebagai kelas tersendiri, bukan dibuang dari evaluasi.

# **LABEL ACUAN KORPUS TAHAP 1**

Tahap ini membangun label acuan untuk klip korpus Tahap 1 — korpus potongan yang
dipakai melatih model Tahap 2 (v12) dan yang klip validasinya ditranskripsikan
oleh notebook 07.

Korpus itu memuat empat bentuk ucapan, dan justru keragaman itulah yang membuatnya
berharga untuk menguji NLU:

| varian | contoh | yang diuji |
|---|---|---|
| `produk` | "abc kecap asin enam ratus dua puluh mililiter" | pengenalan produk tanpa konteks kalimat sama sekali |
| `intent_produk` | "jual abc kecap asin ..." | intent + produk, tanpa jumlah |
| `produk_qty` | "abc kecap asin ... dua bungkus" | produk + jumlah, tanpa intent |
| `full` | kalimat penuh seperti aslinya | seluruh slot sekaligus |

Tiga varian pertama adalah **ucapan sepotong**, dan itulah cara aplikasi
sebenarnya dipakai: kasir menyebut nama produk saja, bukan kalimat lengkap.
Pada varian tersebut sebagian besar slot memang tidak diucapkan, sehingga yang
diuji bukan hanya kemampuan mengisi slot, melainkan juga kemampuan
**tidak mengisi** slot yang tidak ada — kesalahan yang sama seriusnya, karena
jumlah atau harga yang dikarang akan langsung masuk ke catatan transaksi.

Prosedur pelabelannya sama persis dengan bagian sebelumnya, dijalankan atas
transkrip acuan tiap klip.

In [ ]:
acuan_t1 = None
if not MANIFEST_TAHAP1:
    print('[LEWATI] manifest korpus Tahap 1 tidak ditemukan — '
          'kondisi Tahap 1 tidak akan dievaluasi.')
else:
    manifest_t1 = pd.read_csv(MANIFEST_TAHAP1)
    manifest_t1['teks'] = manifest_t1['transcript'].apply(bersihkan_teks)

    acuan_t1 = pd.DataFrame([label_acuan(t) for t in manifest_t1['teks']])
    acuan_t1.insert(0, 'filename', manifest_t1['filename'].values)
    acuan_t1.insert(1, 'teks', manifest_t1['teks'].values)
    acuan_t1['variant'] = manifest_t1['variant'].values
    acuan_t1['kondisi_akustik'] = np.where(manifest_t1['augment'].values == 'clean',
                                           'bersih', 'teraugmentasi')

    print(f'klip korpus Tahap 1 : {len(acuan_t1):,}')
    print(f'produk terjangkar   : {acuan_t1.produk.notna().sum():,} '
          f'({100*acuan_t1.produk.notna().mean():.1f}%)')
    print(f'jumlah terlabeli    : {acuan_t1.jumlah.notna().sum():,} '
          f'({100*acuan_t1.jumlah.notna().mean():.1f}%)')
    print(f'harga terlabeli     : {acuan_t1.harga.notna().sum():,} '
          f'({100*acuan_t1.harga.notna().mean():.1f}%)')

    # Pemeriksaan silang yang sama: harga = harga satuan katalog x jumlah.
    bisa = (acuan_t1.harga.notna() & acuan_t1.jumlah.notna()
            & acuan_t1.produk.notna() & bool(HARGA_SATUAN))
    cek_t1 = acuan_t1[bisa].copy()
    if len(cek_t1):
        cek_t1['harga_hitung'] = cek_t1.apply(
            lambda r: HARGA_SATUAN.get(r['produk'], -1) * r['jumlah'], axis=1)
        cocok_t1 = int((cek_t1.harga == cek_t1.harga_hitung).sum())
        print(f'\nPemeriksaan silang harga: {cocok_t1:,}/{len(cek_t1):,} '
              f'({100*cocok_t1/len(cek_t1):.2f}%)')
    else:
        print('\nTidak ada klip yang menyebut harga sekaligus jumlah — '
              'pemeriksaan silang tidak berlaku di sini.')

    print('\nSlot yang terisi per varian ucapan (% dari klip varian itu):')
    komposisi = acuan_t1.groupby('variant').agg(
        n=('filename', 'size'),
        ada_intent=('intent', lambda x: 100 * (x != 'tanpa_penanda').mean()),
        ada_produk=('produk', lambda x: 100 * x.notna().mean()),
        ada_jumlah=('jumlah', lambda x: 100 * x.notna().mean()),
        ada_harga=('harga', lambda x: 100 * x.notna().mean()),
    ).round(1)
    print(komposisi.to_string())

**Insight:**

Tabel komposisi slot adalah pembacaan terpenting di sini. Varian `produk`
hampir seluruhnya berisi slot kosong kecuali produk, sehingga pada kondisi itu
metrik entitas sebagian besar menilai apakah sistem **menahan diri** — tidak
mengarang jumlah atau harga yang tidak pernah diucapkan.

Porsi harga yang terlabeli juga jauh lebih kecil daripada di dataset asli.
Penyebabnya bukan pelabelan yang gagal, melainkan pembentukan korpusnya: klip
yang melebihi jendela 8 detik dipangkas rentang katanya dari belakang — harga
lebih dulu — supaya label dan audio tetap sepakat. Jadi pada klip itu harga
memang benar-benar tidak terdengar, dan label "tidak ada harga" adalah label
yang benar.

# **RANCANGAN NLU BERBASIS ATURAN**

Bagian ini adalah jawaban atas **RM-4**. Masukannya satu untai teks hasil
transkripsi; keluarannya satu struktur transaksi:

```
"jual indomie goreng dua bungkus enam ribu rupiah"
        |
        v
{ intent: "jual" -> catat_penjualan,
  produk: "indomie goreng", jumlah: 2, harga: 6000 }
```

**Mengapa berbasis aturan, bukan model terlatih.** Domainnya sempit dan
strukturnya diketahui: katalog 121 produk yang tetap, sepuluh kata bilangan,
dan pola kalimat transaksi warung yang terbatas. Pada kondisi seperti ini
aturan memberi tiga hal yang sulit ditandingi model: setiap keputusan dapat
ditelusuri, penambahan produk baru tidak menuntut pelatihan ulang, dan tidak ada
kebutuhan data berlabel *intent/entity* dalam jumlah besar. Batasnya juga jelas
dan disebutkan apa adanya — aturan tidak menggeneralisasi ke kalimat di luar pola
yang dirancang.

**Urutan pemrosesan.** Urutannya bukan sekadar susunan; tiap langkah menyingkirkan
kata yang bila dibiarkan akan disalahartikan langkah berikutnya.

| # | langkah | aturan | alasan urutannya |
|---|---|---|---|
| 1 | normalisasi | huruf kecil, hanya `a-z` dan spasi | menyamakan ruang simbol dengan keluaran model |
| 2 | *intent* | kata pertama dicocokkan ke daftar kata intent; toleransi salah eja lewat kemiripan karakter >= 0.70 | dilakukan lebih dulu agar kata intent tidak terbawa ke nama produk |
| 3 | harga | deretan kata bilangan bersambung tepat sebelum `rupiah`; bila `rupiah` tidak ada, deretan bilangan di ujung ucapan yang berakhir `ribu`/`juta` | disingkirkan sebelum jumlah dicari, karena nominal harga penuh kata bilangan ("dua puluh empat ribu"). Penanda cadangan diperlukan karena pada klip sepotong kata `rupiah` kerap terpangkas sementara nominalnya masih terucap |
| 4 | jumlah | kata bilangan + satuan hitung; atau kata bilangan di ujung ucapan; atau kata bilangan di depan nama produk | satuan hitung membedakan "dua bungkus" (jumlah) dari "enam ratus mililiter" (ukuran kemasan) |
| 4b | penjagaan | bila sisa kata persis sama dengan sebuah nama katalog, jumlah dianggap tidak diucapkan | nama produk boleh diawali kata bilangan ("dua kelinci kacang"); tanpa penjagaan ini kata pertamanya hilang |
| 5 | produk | sisa kata setelah langkah 2-4 dibuang | yang tersisa memang bagian nama produk |
| 6 | pencocokan | cocok persis ke katalog; bila tidak, kemiripan campuran kata + karakter; di bawah ambang -> "tidak dikenali" | teks ASR mengandung salah eja, sehingga pencocokan persis saja tidak cukup |

**Dua sumber harga yang dibedakan.** `harga_ucapan` adalah nominal yang benar-benar
disebutkan penutur; `harga_dihitung` adalah `harga satuan katalog x jumlah`.
Aplikasi memakai yang kedua karena selalu tersedia dan tidak bergantung pada
ketepatan transkripsi angka, tetapi evaluasi entitas harga menilai yang pertama —
itulah yang benar-benar diekstraksi dari teks.

In [ ]:
class NLUAturan:
    """Pemahaman makna berbasis aturan untuk perintah suara transaksi warung.

    Dirancang untuk bekerja di atas teks hasil transkripsi yang mungkin
    mengandung salah eja, bukan hanya di atas teks yang sempurna.
    """

    def __init__(self, katalog, harga_satuan, ambang=AMBANG_PRODUK,
                 ambang_intent=0.70):
        self.katalog = list(katalog)
        self.nama_produk = set(self.katalog)
        # Tabel harga boleh kosong: ia hanya dipakai menghitung harga_dihitung,
        # bukan untuk mengenali produk.
        self.harga_satuan = harga_satuan or {}
        self.ambang = ambang
        self.ambang_intent = ambang_intent

    # ── Langkah 2 ────────────────────────────────────────────────
    def deteksi_intent(self, kata_kata):
        """Kata pertama -> intent. Toleran terhadap salah eja ringan dari ASR.

        Model kerap menuliskan "jual" sebagai "juai" atau "jal". Menolak ejaan
        seperti itu berarti membuang intent yang sebenarnya terdengar jelas,
        jadi kecocokan persis dicoba dulu, lalu kemiripan karakter.
        """
        if not kata_kata:
            return 'tanpa_penanda', 0.0
        awal = kata_kata[0]
        if awal in KATA_INTENT:
            return awal, 1.0
        terbaik, skor = None, 0.0
        for kata in KATA_INTENT:
            s = SequenceMatcher(None, awal, kata).ratio()
            if s > skor:
                terbaik, skor = kata, s
        if skor >= self.ambang_intent:
            return terbaik, skor
        return 'tanpa_penanda', 0.0

    # ── Langkah 3 ────────────────────────────────────────────────
    def ekstrak_harga(self, kata_kata):
        """Nominal yang diucapkan -> (harga, indeks_awal, indeks_akhir_eksklusif).

        Penanda utamanya kata 'rupiah'; nominalnya adalah deretan kata bilangan
        yang bersambung tepat sebelum penanda itu.

        Penanda cadangan: ucapan yang berakhir 'ribu'/'juta' tanpa kata
        'rupiah'. Ini bentuk yang muncul pada klip korpus Tahap 1, yang kata
        'rupiah'-nya terpangkas batas jendela 8 detik, dan juga pada transkripsi
        yang terpotong di ujung. Aturannya sama dengan yang dipakai pelabelan
        acuan, sehingga keduanya menilai hal yang sama.
        """
        if 'rupiah' in kata_kata:
            akhir = len(kata_kata) - 1 - kata_kata[::-1].index('rupiah')
            awal = akhir - 1
            while awal >= 0 and kata_kata[awal] in KATA_ANGKA:
                awal -= 1
            awal += 1
            if awal >= akhir:
                return None, akhir, akhir + 1   # 'rupiah' tanpa nominal
            return teks_ke_angka(kata_kata[awal:akhir]), awal, akhir + 1

        if kata_kata and kata_kata[-1] in ('ribu', 'juta'):
            awal = len(kata_kata) - 1
            while awal >= 0 and kata_kata[awal] in KATA_ANGKA:
                awal -= 1
            awal += 1
            if awal < len(kata_kata):
                return teks_ke_angka(kata_kata[awal:]), awal, len(kata_kata)

        return None, -1, -1

    # ── Langkah 4 ────────────────────────────────────────────────
    def ekstrak_jumlah(self, kata_kata, batas_harga):
        """Banyaknya barang -> (jumlah, indeks_awal, indeks_akhir_eksklusif).

        Tiga aturan, diterapkan berurutan. Kata bilangan yang diikuti kata skala
        ('ratus', 'puluh', 'ribu') atau satuan ukuran ('mililiter', 'gram')
        selalu dilewati: itu bagian nama produk atau nominal harga, bukan jumlah.
        """
        n = len(kata_kata)
        batas = batas_harga if batas_harga >= 0 else n

        def boleh(i):
            if kata_kata[i] not in BILANGAN or i >= batas:
                return False
            sesudah = kata_kata[i + 1] if i + 1 < n else None
            return sesudah not in SKALA and sesudah not in SATUAN_UKURAN

        # R1: kata bilangan + satuan hitung -> "dua bungkus"
        for i in range(batas):
            if boleh(i) and i + 1 < n and kata_kata[i + 1] in SATUAN_HITUNG:
                return BILANGAN[kata_kata[i]], i, i + 2

        # R2: kata bilangan di ujung ucapan -> "teh kotak dua ratus mililiter dua"
        if batas > 0 and boleh(batas - 1):
            return BILANGAN[kata_kata[batas - 1]], batas - 1, batas

        # R3: kata bilangan di depan nama produk -> "empat glad two glow moisturizer"
        if boleh(0):
            return BILANGAN[kata_kata[0]], 0, 1

        return None, -1, -1

    # ── Langkah 6 ────────────────────────────────────────────────
    def cocokkan_produk(self, span, gerbang=True):
        """Span teks -> (nama produk katalog, skor kemiripan 0..1).

        gerbang=False mengembalikan kandidat terbaik apa adanya tanpa menerapkan
        ambang, dipakai untuk analisis sensitivitas ambang.

        Skor menggabungkan dua ukuran yang saling menutupi kelemahan:
          - recall kata  : tahan terhadap kata yang hilang dari transkrip
          - kemiripan karakter : tahan terhadap salah eja ("kateap" vs "kecap")
        Bobot 0.6/0.4 mengikuti jalur pencocokan pada layanan REST FinSense,
        supaya evaluasi ini menilai aturan yang benar-benar dipakai aplikasi.
        """
        span = span.strip()
        if not span:
            return None, 0.0
        if span in self.nama_produk:
            return span, 1.0
        kata_span = set(span.split())
        terbaik, skor_terbaik = None, 0.0
        for nama in self.katalog:
            kata_nama = nama.split()
            recall = len(set(kata_nama) & kata_span) / max(len(kata_nama), 1)
            mirip = SequenceMatcher(None, span, nama).ratio()
            skor = 0.6 * recall + 0.4 * mirip
            if skor > skor_terbaik:
                terbaik, skor_terbaik = nama, skor
        if gerbang and skor_terbaik < self.ambang:
            return None, skor_terbaik
        return terbaik, skor_terbaik

    # ── Alur utama ───────────────────────────────────────────────
    def parse(self, teks):
        kata_kata = bersihkan_teks(teks).split()
        intent, skor_intent = self.deteksi_intent(kata_kata)
        harga, h0, h1 = self.ekstrak_harga(kata_kata)

        # Penjagaan sebelum aturan jumlah dijalankan: bila sisa kata setelah
        # kata intent dan nominal harga PERSIS sama dengan sebuah nama di
        # katalog, berarti tidak ada jumlah yang diucapkan sama sekali.
        #
        # Tanpa ini, produk yang namanya diawali kata bilangan kehilangan kata
        # pertamanya: "dua kelinci kacang" terbaca sebagai 2 buah "kelinci
        # kacang". Kasus ini ditemukan saat menguji korpus Tahap 1 pada varian
        # ucapan `produk`, dan hanya terlihat di sana karena pada kalimat penuh
        # aturan satuan hitung sudah lebih dulu menemukan jumlah yang benar.
        sisa_idx = [i for i in range(len(kata_kata))
                    if not (i == 0 and intent != 'tanpa_penanda')
                    and not (h0 >= 0 and h0 <= i < h1)]
        sisa_teks = ' '.join(kata_kata[i] for i in sisa_idx)
        if sisa_teks in self.nama_produk:
            jumlah, j0, j1 = None, -1, -1
        else:
            jumlah, j0, j1 = self.ekstrak_jumlah(kata_kata, h0)

        # Langkah 5: sisakan kata yang bukan intent, bukan jumlah, bukan harga.
        buang = set()
        if intent != 'tanpa_penanda' and kata_kata:
            buang.add(0)
        if j0 >= 0:
            buang.update(range(j0, j1))
        if h0 >= 0:
            buang.update(range(h0, h1))
        span = ' '.join(w for i, w in enumerate(kata_kata) if i not in buang)

        kandidat, skor_produk = self.cocokkan_produk(span, gerbang=False)
        produk = kandidat if skor_produk >= self.ambang else None
        satuan = self.harga_satuan.get(produk) if produk else None
        harga_dihitung = satuan * jumlah if satuan is not None and jumlah else None

        return {
            'intent': intent,
            'intent_kanonik': INTENT_KANONIK.get(intent, 'catat_penjualan'),
            'skor_intent': round(skor_intent, 3),
            'produk': produk,
            # Kandidat terbaik TANPA memandang ambang. Disimpan agar pengaruh
            # ambang dapat dianalisis tanpa mengurai ulang seluruh data.
            'produk_kandidat': kandidat,
            'skor_produk': round(skor_produk, 3),
            'jumlah': jumlah,
            'harga': harga,                    # nominal yang diucapkan
            'harga_dihitung': harga_dihitung,  # harga satuan x jumlah
            'span_produk': span,
        }


nlu = NLUAturan(PRODUK_URUT, HARGA_SATUAN)
print('NLU berbasis aturan siap:', len(PRODUK_URUT), 'produk katalog,',
      f'ambang produk {AMBANG_PRODUK}')

**Insight:**

Aturan R1 pada `ekstrak_jumlah` adalah yang paling menentukan. Tanpa syarat
satuan hitung, kata bilangan di dalam nama produk akan terbaca sebagai jumlah:
"aqua enam ratus mililiter" menjadi enam botol, dan nominal transaksi menjadi
enam kali lipat. Karena itu kata bilangan yang diikuti kata skala atau satuan
ukuran selalu dilewati — satu syarat kecil yang mencegah kelas kesalahan yang
sulit dilihat dari metrik agregat.

Toleransi salah eja pada `deteksi_intent` dirancang untuk teks ASR, bukan untuk
teks acuan. Pada teks acuan ia tidak pernah aktif; pengaruhnya baru terlihat
pada kondisi ASR, dan di situlah ia dinilai.

# **DEMONSTRASI PARSER**

Tahap ini menjalankan parser pada beberapa ucapan, termasuk teks hasil
transkripsi yang mengandung kesalahan, untuk memperlihatkan perilakunya sebelum
metrik dihitung.

In [ ]:
contoh_uji = [
    'jual abc kecap asin enam ratus dua puluh mililiter dua bungkus tiga puluh enam ribu rupiah',
    'tolong indomie goreng lima bungkus',
    'teh kotak dua ratus mililiter dua',
    'empat glad two glow moisturizer',
    'beli aqua enam ratus mililiter tiga dus',
    'juai indomie gorng dua bungkus',      # teks ASR: intent & produk salah eja
    'jual baygon obat nyamuk bakar sepuluh bungkus tuh puh duh lib rbu rupiah',  # harga rusak
    'assalamualaikum pak mau tanya',       # bukan perintah transaksi
    'abc kecap asin enam ratus dua puluh mililiter',   # klip Tahap 1: produk saja
    'dua kelinci kacang',                  # nama produk diawali kata bilangan
    'dua kelinci kacang tiga bungkus',     # nama produk + jumlah sesungguhnya
    'jual abc kecap asin enam ratus dua puluh mililiter satu bungkus delapan belas ribu',  # 'rupiah' terpangkas
]

for t in contoh_uji:
    h = nlu.parse(t)
    print(f'teks    : {t}')
    print(f'  intent: {h["intent"]:14s} -> {h["intent_kanonik"]}')
    print(f'  produk: {h["produk"]} (skor {h["skor_produk"]}, span "{h["span_produk"]}")')
    print(f'  jumlah: {h["jumlah"]} | harga diucapkan: {h["harga"]} | harga dihitung: {h["harga_dihitung"]}')
    print()

**Insight:**

Dua contoh terakhir sengaja dipilih karena memperlihatkan batas rancangan ini.

Pada ucapan dengan nominal harga yang rusak, `harga` (nominal yang diucapkan)
menjadi salah atau kosong, tetapi `harga_dihitung` tetap benar selama produk dan
jumlahnya terbaca. Inilah alasan aplikasi memakai harga hitungan, bukan harga
ucapan: kesalahan transkripsi pada deretan angka tidak merambat menjadi nominal
transaksi yang keliru.

Pada ucapan yang bukan perintah transaksi, skor produk berada di bawah ambang
sehingga hasilnya "tidak dikenali". Perilaku menolak ini yang membuat sistem
aman dipakai: tanpa ambang, setiap ucapan apa pun akan tetap dipetakan ke salah
satu dari 121 produk.

# **METRIK EVALUASI**

Tahap ini mendefinisikan seluruh metrik yang dipakai. Semuanya dihitung dari
perbandingan label acuan dengan keluaran parser, tanpa konstanta.

**Klasifikasi intent.**

$$\text{Akurasi} = \frac{\text{prediksi benar}}{\text{seluruh ucapan}}, \qquad
F1 = 2\cdot\frac{P \cdot R}{P + R}$$

Dilaporkan dua rata-rata F1 yang berbeda maknanya pada kelas tak seimbang:
*macro* merata-ratakan F1 tiap kelas dengan bobot sama (kelas kecil ikut
menentukan), *weighted* menimbang menurut jumlah data tiap kelas (mengikuti
sebaran nyata).

**Ekstraksi entitas** dinilai sebagai *slot filling* dengan kecocokan persis:

- **TP** — entitas ada pada acuan dan nilainya sama dengan prediksi
- **FP** — sistem mengisi entitas padahal acuannya kosong, atau mengisinya salah
- **FN** — entitas ada pada acuan tetapi tidak diisi, atau diisi salah

Pengisian yang salah dihitung sebagai FP **dan** FN sekaligus; ini konvensi baku
pada evaluasi *slot filling*, dan membuat sistem yang asal menebak tidak bisa
mendapat nilai bagus.

**Loss.** Dua ukuran yang berlaku untuk pengklasifikasi deterministik:

$$L_{0\text{-}1} = 1 - \text{Akurasi}, \qquad
L_{\text{Hamming}} = \frac{1}{N \cdot S}\sum_{i}\sum_{s} \mathbb{1}[\hat{y}_{is} \neq y_{is}]$$

dengan $S = 4$ slot (intent, produk, jumlah, harga). Keduanya diturunkan dari
perbandingan label, bukan dari probabilitas, sehingga tidak memerlukan keluaran
probabilistik yang memang tidak dimiliki NLU berbasis aturan.

**Akurasi gabungan (*joint accuracy*)** — persentase ucapan yang intent dan
seluruh entitasnya benar sekaligus. Inilah metrik yang paling dekat dengan
pengalaman kasir: satu slot salah berarti transaksi tetap harus dikoreksi.

In [ ]:
SLOT = ['produk', 'jumlah', 'harga']


def samakan(nilai):
    """Samakan bentuk nilai slot agar perbandingan tidak terpeleset tipe data."""
    if nilai is None or (isinstance(nilai, float) and np.isnan(nilai)):
        return None
    if isinstance(nilai, (int, np.integer)):
        return int(nilai)
    if isinstance(nilai, float):
        return int(nilai)
    return str(nilai).strip()


def metrik_slot(acuan_nilai, prediksi_nilai):
    """Hitung TP/FP/FN satu slot untuk seluruh ucapan -> presisi, recall, F1."""
    tp = fp = fn = benar_kosong = 0
    for a, p in zip(acuan_nilai, prediksi_nilai):
        a, p = samakan(a), samakan(p)
        if a is None and p is None:
            benar_kosong += 1
        elif a is None and p is not None:
            fp += 1
        elif a is not None and p is None:
            fn += 1
        elif a == p:
            tp += 1
        else:
            fp += 1; fn += 1
    if tp + fp + fn == 0:
        # Slot tidak pernah muncul di acuan maupun prediksi. Presisi, recall,
        # dan F1 TIDAK terdefinisi di sini; melaporkannya sebagai 0 akan
        # terbaca sebagai kegagalan total, padahal tidak ada yang gagal —
        # tidak ada yang diuji. NaN membuatnya terlihat apa adanya.
        presisi = recall = f1 = float('nan')
    else:
        presisi = tp / max(tp + fp, 1)
        recall = tp / max(tp + fn, 1)
        f1 = 2 * presisi * recall / max(presisi + recall, 1e-12)
    return {'TP': tp, 'FP': fp, 'FN': fn, 'benar_kosong': benar_kosong,
            'presisi': presisi, 'recall': recall, 'f1': f1,
            'terdefinisi': tp + fp + fn > 0,
            'akurasi_slot': (tp + benar_kosong) / max(len(acuan_nilai), 1)}


def evaluasi_kondisi(nama, acuan_df, prediksi_df):
    """Seluruh metrik satu kondisi masukan -> (ringkasan, tabel slot, tabel per kelas)."""
    y_acuan = acuan_df['intent'].astype(str).values
    y_prediksi = prediksi_df['intent'].astype(str).values
    kelas = sorted(set(y_acuan) | set(y_prediksi))

    akurasi = accuracy_score(y_acuan, y_prediksi)
    p_mac, r_mac, f_mac, _ = precision_recall_fscore_support(
        y_acuan, y_prediksi, average='macro', zero_division=0)
    p_wtd, r_wtd, f_wtd, _ = precision_recall_fscore_support(
        y_acuan, y_prediksi, average='weighted', zero_division=0)

    akurasi_kanonik = accuracy_score(acuan_df['intent_kanonik'].astype(str),
                                     prediksi_df['intent_kanonik'].astype(str))

    # Pembanding wajib untuk akurasi: seberapa jauh angka itu di atas menebak
    # kelas terbanyak. Tanpa ini, himpunan uji yang hanya berisi satu kelas
    # akan melaporkan akurasi 100% yang tidak berarti apa-apa.
    frekuensi = collections.Counter(y_acuan)
    baseline = max(frekuensi.values()) / max(len(y_acuan), 1)
    n_kelas = len(frekuensi)

    # Seberapa sering kata pertama — penentu intent — rusak pada teks masukan.
    # Inilah ukuran kesulitan sebenarnya dari tugas klasifikasi intent di sini.
    kata1_rusak = 0
    if 'teks' in acuan_df.columns and 'teks_masukan' in prediksi_df.columns:
        for teks_acu, teks_in in zip(acuan_df['teks'].astype(str),
                                     prediksi_df['teks_masukan'].astype(str)):
            a1 = teks_acu.split()[0] if teks_acu.split() else ''
            b1 = teks_in.split()[0] if teks_in.split() else ''
            kata1_rusak += int(a1 != b1)

    slot = {s: metrik_slot(acuan_df[s].values, prediksi_df[s].values) for s in SLOT}
    # F1 mikro entitas menjumlahkan TP/FP/FN lintas slot, jadi slot yang tidak
    # terdefinisi (tidak pernah muncul) otomatis tidak mempengaruhinya.
    tp = sum(slot[s]['TP'] for s in SLOT)
    fp = sum(slot[s]['FP'] for s in SLOT)
    fn = sum(slot[s]['FN'] for s in SLOT)
    p_mikro = tp / max(tp + fp, 1)
    r_mikro = tp / max(tp + fn, 1)
    f_mikro = 2 * p_mikro * r_mikro / max(p_mikro + r_mikro, 1e-12)

    # Hamming loss atas 4 slot: intent + tiga entitas.
    salah = 0
    benar_semua = 0
    for i in range(len(acuan_df)):
        cocok = [y_acuan[i] == y_prediksi[i]]
        cocok += [samakan(acuan_df[s].values[i]) == samakan(prediksi_df[s].values[i])
                  for s in SLOT]
        salah += sum(1 for c in cocok if not c)
        benar_semua += int(all(cocok))
    hamming = salah / max(len(acuan_df) * 4, 1)

    ringkas = {
        'kondisi': nama,
        'n_ucapan': len(acuan_df),
        'akurasi_intent': akurasi,
        'akurasi_intent_kanonik': akurasi_kanonik,
        'n_kelas_intent': n_kelas,
        'baseline_mayoritas': baseline,
        'kata1_rusak': kata1_rusak / max(len(acuan_df), 1),
        'f1_macro_intent': f_mac,
        'f1_weighted_intent': f_wtd,
        'presisi_macro_intent': p_mac,
        'recall_macro_intent': r_mac,
        'f1_mikro_entitas': f_mikro,
        'presisi_mikro_entitas': p_mikro,
        'recall_mikro_entitas': r_mikro,
        'f1_produk': slot['produk']['f1'],
        'f1_jumlah': slot['jumlah']['f1'],
        'f1_harga': slot['harga']['f1'],
        'akurasi_gabungan': benar_semua / max(len(acuan_df), 1),
        'loss_0_1_intent': 1 - akurasi,
        'loss_hamming': hamming,
    }

    tabel_slot = pd.DataFrame([{'kondisi': nama, 'slot': s, **slot[s]} for s in SLOT])
    laporan = classification_report(y_acuan, y_prediksi, labels=kelas,
                                    zero_division=0, output_dict=True)
    per_kelas = pd.DataFrame([
        {'kondisi': nama, 'intent': k, 'presisi': v['precision'], 'recall': v['recall'],
         'f1': v['f1-score'], 'dukungan': int(v['support'])}
        for k, v in laporan.items() if k in kelas
    ])
    return ringkas, tabel_slot, per_kelas


# Uji kewarasan fungsi metrik pada kasus yang jawabannya sudah diketahui.
_a = pd.DataFrame({'produk': ['aqua', 'teh', None], 'jumlah': [1, 2, 3], 'harga': [None, 100, 200]})
_p = pd.DataFrame({'produk': ['aqua', 'kopi', 'susu'], 'jumlah': [1, 2, None], 'harga': [None, 100, 999]})
_m = metrik_slot(_a['produk'].values, _p['produk'].values)
assert (_m['TP'], _m['FP'], _m['FN']) == (1, 2, 1), _m
print('Fungsi metrik slot lolos uji: TP/FP/FN terhitung sesuai konvensi slot filling.')

**Insight:**

Slot yang tidak pernah muncul sama sekali — misalnya harga pada klip yang
memang tidak menyebut nominal — dilaporkan `NaN`, bukan 0. Keduanya berbeda
artinya: 0 berarti sistem selalu salah, `NaN` berarti tidak ada yang diuji.

Perhatikan pula perlakuan terhadap slot kosong. Ucapan yang memang tidak menyebut
harga bukan kegagalan sistem bila harganya tidak diisi; kasus itu dicatat
sebagai `benar_kosong` dan tidak masuk hitungan F1. Tanpa pemisahan ini, F1
harga akan tampak buruk hanya karena sebagian besar pedagang memang tidak
menyebut nominal saat bicara — dan angka yang salah arah seperti itu akan
mengarahkan perbaikan ke tempat yang keliru.

# **PENYIAPAN KONDISI MASUKAN**

Tahap ini menyiapkan tiga kondisi masukan. Label acuannya **selalu sama** —
diturunkan dari transkrip manusia — sehingga perbedaan angka antar kondisi
murni berasal dari mutu teks yang masuk ke NLU, bukan dari acuan yang berbeda.

Teks ASR dibaca dari keluaran notebook 07 agar kedua notebook menilai
transkripsi yang sama persis.

In [ ]:
# Tiap kondisi memakai label acuan milik korpusnya sendiri. Nama berkas pada
# korpus Tahap 1 berbeda dari finsense_expanded, jadi memakai satu indeks acuan
# untuk semuanya akan menghasilkan irisan kosong tanpa peringatan apa pun.
ACUAN_SUMBER = {'teks_acuan': acuan.set_index('filename')}
if acuan_t1 is not None:
    ACUAN_SUMBER['teks_acuan_tahap1'] = acuan_t1.set_index('filename')

SUMBER_ACUAN = {
    'teks_acuan': 'teks_acuan',
    'teks_acuan_tahap1': 'teks_acuan_tahap1',
    'asr_speaker_dependent': 'teks_acuan',
    'asr_speaker_independent': 'teks_acuan',
    'asr_tahap1_val': 'teks_acuan_tahap1',
}

KONDISI = {}
KONDISI['teks_acuan'] = pd.DataFrame({
    'filename': df['filename'].values,
    'teks_masukan': df['teks'].values,
})
if acuan_t1 is not None:
    KONDISI['teks_acuan_tahap1'] = pd.DataFrame({
        'filename': acuan_t1['filename'].values,
        'teks_masukan': acuan_t1['teks'].values,
    })

for nama, path in BERKAS_ASR.items():
    if not os.path.exists(path):
        print(f'[LEWATI] {nama}: {path} belum ada — jalankan notebook 07 lebih dulu.')
        continue
    sumber = SUMBER_ACUAN.get(nama, 'teks_acuan')
    if sumber not in ACUAN_SUMBER:
        print(f'[LEWATI] {nama}: label acuan "{sumber}" tidak tersedia.')
        continue
    idx_acuan = ACUAN_SUMBER[sumber]
    dfa = pd.read_csv(path)
    dfa['teks_prediksi'] = dfa['teks_prediksi'].fillna('')
    # Hanya ucapan yang label acuannya tersedia yang dapat dinilai.
    dfa = dfa[dfa['filename'].isin(idx_acuan.index)]
    KONDISI[nama] = pd.DataFrame({
        'filename': dfa['filename'].values,
        'teks_masukan': dfa['teks_prediksi'].values,
    })

for nama, d in KONDISI.items():
    print(f'{nama:26s}: {len(d):,} ucapan  (acuan: {SUMBER_ACUAN.get(nama, "teks_acuan")})')

**Insight:**

Kondisi ASR memuat lebih sedikit ucapan daripada kondisi teks acuan, karena
notebook 07 hanya mentranskripsikan data uji tiap skenario. Perbandingan
antar-kondisi karena itu tidak dilakukan pada ucapan yang persis sama.
Bagian *Analisis kesalahan* menangani hal ini dengan membandingkan kedua kondisi
**pada irisan ucapan yang sama**, sehingga selisihnya benar-benar disebabkan
kualitas teks, bukan komposisi sampel.

# **EVALUASI: AKURASI DAN F1-SCORE**

Tahap ini menjalankan NLU pada tiap kondisi lalu menghitung seluruh metrik yang
telah didefinisikan.

In [ ]:
HASIL_NLU = {}
PREDIKSI = {}

for nama, d in KONDISI.items():
    t0 = time.time()
    hasil = [nlu.parse(t) for t in d['teks_masukan']]
    prediksi = pd.DataFrame(hasil)
    prediksi.insert(0, 'filename', d['filename'].values)
    prediksi.insert(1, 'teks_masukan', d['teks_masukan'].values)

    idx_acuan = ACUAN_SUMBER[SUMBER_ACUAN.get(nama, 'teks_acuan')]
    acuan_selaras = idx_acuan.loc[d['filename'].values].reset_index()
    PREDIKSI[nama] = (acuan_selaras, prediksi)

    lewat = time.time() - t0
    print(f'{nama:26s}: {len(d):,} ucapan diurai dalam {lewat:.1f}s '
          f'({1000*lewat/max(len(d),1):.2f} ms/ucapan)')

ringkasan, tabel_slot_semua, tabel_kelas_semua = [], [], []
for nama, (a, p) in PREDIKSI.items():
    r, ts, tk = evaluasi_kondisi(nama, a, p)
    ringkasan.append(r); tabel_slot_semua.append(ts); tabel_kelas_semua.append(tk)

tabel_ringkas = pd.DataFrame(ringkasan)
tabel_slot = pd.concat(tabel_slot_semua, ignore_index=True)
tabel_kelas = pd.concat(tabel_kelas_semua, ignore_index=True)

tampil = tabel_ringkas[['kondisi', 'n_ucapan', 'n_kelas_intent', 'baseline_mayoritas',
                        'akurasi_intent', 'kata1_rusak', 'f1_macro_intent',
                        'f1_mikro_entitas', 'f1_produk', 'f1_jumlah', 'f1_harga',
                        'akurasi_gabungan']].copy()
for k in tampil.columns[3:]:
    tampil[k] = (100 * tampil[k]).round(2)
print(tampil.to_string(index=False))

# ─── Pemeriksaan kesahihan angka intent ───
print()
for _, r in tabel_ringkas.iterrows():
    if r['n_kelas_intent'] < 2:
        print(f'PERINGATAN [{r["kondisi"]}]: data ujinya hanya memuat SATU kelas intent, '
              f'sehingga akurasi {100*r["akurasi_intent"]:.2f}% sama dengan menebak kelas itu '
              'terus-menerus. Angka intent untuk kondisi ini tidak boleh dilaporkan '
              'sebagai capaian.')
    elif r['akurasi_intent'] <= r['baseline_mayoritas'] + 1e-9:
        print(f'PERINGATAN [{r["kondisi"]}]: akurasi intent tidak melampaui baseline '
              f'kelas mayoritas ({100*r["baseline_mayoritas"]:.2f}%).')
    if r['kata1_rusak'] == 0 and r['kondisi'].startswith('asr_'):
        print(f'CATATAN [{r["kondisi"]}]: kata pertama tidak pernah rusak pada teks ASR, '
              'jadi klasifikasi intent di sini praktis tidak diuji oleh kesalahan transkripsi.')
tabel_ringkas.to_csv(os.path.join(KELUARAN, 'metrik_nlu.csv'), index=False)
tabel_slot.to_csv(os.path.join(KELUARAN, 'metrik_slot.csv'), index=False)
tabel_kelas.to_csv(os.path.join(KELUARAN, 'metrik_per_intent.csv'), index=False)
tampil

**Insight:**

**Baca angka kondisi "teks acuan" dengan hati-hati.** Pada teks yang bersih,
label acuan dan parser sama-sama berpijak pada katalog produk yang sama, sehingga
nilai mendekati 100% sebagian besar merupakan akibat konstruksi, bukan bukti
keunggulan. Fungsinya hanya satu: memastikan aturan tidak cacat — kalau di teks
sempurna pun aturannya gagal, kesalahan ada pada aturannya. Angka yang benar-benar
mengukur performansi sistem adalah kondisi ASR, dan itulah yang seharusnya
dikutip sebagai jawaban RM-5.

Tiga angka pantas dibaca berurutan.

`akurasi_intent` dan `f1_macro_intent` biasanya berjauhan, dan itu wajar: akurasi
didominasi kelas besar (`jual`), sedangkan macro-F1 memberi bobot sama kepada
kelas kecil. Bila keduanya berjauhan, artinya kelas kecil ditangani lebih buruk,
dan hanya macro-F1 yang menunjukkannya.

`f1_mikro_entitas` merangkum ketiga entitas sekaligus, tetapi `f1_produk`,
`f1_jumlah`, dan `f1_harga` yang menunjukkan di mana kesalahan berada. Pada teks
ASR, entitas harga hampir selalu paling rendah — deretan kata bilangan panjang
adalah bagian ucapan yang paling mudah rusak oleh kesalahan transkripsi, dan satu
kata saja yang meleset sudah mengubah nominalnya.

`akurasi_gabungan` adalah angka yang paling dekat dengan pengalaman kasir dan
selalu paling rendah. Angka inilah, bukan akurasi intent, yang menjawab "berapa
persen ucapan yang langsung tercatat benar tanpa koreksi".

# **VISUALISASI HASIL**

Tahap ini menyajikan hasil dalam tiga tampilan: perbandingan metrik utama antar
kondisi, F1 per entitas, dan F1 per kelas intent.

In [ ]:
kondisi_urut = list(PREDIKSI.keys())
label_kondisi = {
    'teks_acuan': 'Teks acuan',
    'teks_acuan_tahap1': 'Teks acuan\nTahap 1 (sepotong)',
    'asr_speaker_dependent': 'ASR v11\nspeaker-dependent',
    'asr_speaker_independent': 'ASR v11\nspeaker-independent',
    'asr_tahap1_val': 'ASR v12\nTahap 1 (val)',
}
nama_tampil = [label_kondisi.get(k, k) for k in kondisi_urut]

fig, ax = plt.subplots(1, 3, figsize=(16, 4.4))

# (a) metrik utama
metrik = ['akurasi_intent', 'f1_macro_intent', 'f1_mikro_entitas', 'akurasi_gabungan']
judul_metrik = ['akurasi intent', 'F1 macro intent', 'F1 mikro entitas', 'akurasi gabungan']
lebar = 0.8 / len(metrik)
posisi = np.arange(len(kondisi_urut))
for i, (m, j) in enumerate(zip(metrik, judul_metrik)):
    nilai = [100 * tabel_ringkas.loc[tabel_ringkas.kondisi == k, m].iloc[0] for k in kondisi_urut]
    bar = ax[0].bar(posisi + i * lebar, nilai, lebar, label=j)
    ax[0].bar_label(bar, fmt='%.1f', fontsize=7, padding=2)
ax[0].set_xticks(posisi + 0.4 - lebar / 2)
ax[0].set_xticklabels(nama_tampil, fontsize=8)
ax[0].set_ylabel('nilai (%)'); ax[0].set_ylim(0, 108)
ax[0].set_title('(a) Metrik utama per kondisi masukan')
ax[0].legend(fontsize=7); ax[0].grid(axis='y', alpha=0.3)

# (b) F1 per entitas
slot_metrik = ['f1_produk', 'f1_jumlah', 'f1_harga']
lebar = 0.8 / len(slot_metrik)
for i, m in enumerate(slot_metrik):
    mentah = [100 * tabel_ringkas.loc[tabel_ringkas.kondisi == k, m].iloc[0] for k in kondisi_urut]
    # Slot yang tidak terdefinisi (tidak pernah muncul di acuan) digambar nol
    # tetapi diberi label 'n/a', supaya batang kosong tidak terbaca sebagai
    # kegagalan.
    nilai = [v if np.isfinite(v) else 0 for v in mentah]
    teks_label = [f'{v:.1f}' if np.isfinite(v) else 'n/a' for v in mentah]
    bar = ax[1].bar(posisi + i * lebar, nilai, lebar, label=m.replace('f1_', ''))
    ax[1].bar_label(bar, labels=teks_label, fontsize=7, padding=2)
ax[1].set_xticks(posisi + 0.4 - lebar / 2)
ax[1].set_xticklabels(nama_tampil, fontsize=8)
ax[1].set_ylabel('F1 (%)'); ax[1].set_ylim(0, 108)
ax[1].set_title('(b) F1 per entitas')
ax[1].legend(fontsize=7); ax[1].grid(axis='y', alpha=0.3)

# (c) F1 per kelas intent
kelas_urut = sorted(tabel_kelas.intent.unique())
lebar = 0.8 / len(kondisi_urut)
posisi_k = np.arange(len(kelas_urut))
for i, k in enumerate(kondisi_urut):
    sub = tabel_kelas[tabel_kelas.kondisi == k].set_index('intent')
    nilai = [100 * sub.loc[c, 'f1'] if c in sub.index else 0 for c in kelas_urut]
    ax[2].bar(posisi_k + i * lebar, nilai, lebar, label=label_kondisi.get(k, k).replace('\n', ' '))
ax[2].set_xticks(posisi_k + 0.4 - lebar / 2)
ax[2].set_xticklabels(kelas_urut, rotation=30, ha='right', fontsize=8)
ax[2].set_ylabel('F1 (%)'); ax[2].set_ylim(0, 108)
ax[2].set_title('(c) F1 per kelas intent')
ax[2].legend(fontsize=7); ax[2].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(KELUARAN, 'metrik_nlu.png'), dpi=150)
plt.show()

**Insight:**

Jarak antara batang "Teks acuan" dan batang "ASR speaker-independent" pada panel
(a) adalah **biaya kesalahan transkripsi** terhadap pemahaman makna. Jarak itu
hampir selalu lebih besar daripada yang diduga dari CER: CER 5% tidak berarti
pemahaman turun 5%, sebab satu karakter yang meleset pada kata bilangan dapat
menggagalkan seluruh entitas jumlah.

Panel (b) menunjukkan bahwa ketiga entitas tidak terdampak sama. Entitas produk
paling tahan karena pencocokan ke katalog dapat memperbaiki ejaan yang rusak;
entitas jumlah dan harga tidak punya penyangga serupa — tidak ada "katalog
angka" yang bisa dijadikan acuan pembetulan.

# **RINCIAN PER BENTUK UCAPAN (KORPUS TAHAP 1)**

Tahap ini membedah hasil kondisi Tahap 1 menurut **bentuk ucapan** dan
**kondisi akustik**. Inilah rincian yang paling menentukan bagi FinSense, sebab
di lapangan kasir jarang mengucapkan kalimat lengkap: yang lazim adalah nama
produk saja, kadang disusul jumlah.

Setiap slot dinilai benar bila nilainya sama dengan acuan, termasuk ketika acuan
menyatakan slot itu **tidak diucapkan** — dalam hal itu sistem dinyatakan benar
hanya bila ia juga tidak mengisinya.

In [ ]:
kondisi_t1 = [k for k in kondisi_urut if SUMBER_ACUAN.get(k) == 'teks_acuan_tahap1']

if not kondisi_t1:
    print('Kondisi korpus Tahap 1 tidak tersedia — bagian ini dilewati.')
    tabel_varian = None
else:
    peta_varian = acuan_t1.set_index('filename')['variant']
    peta_akustik = acuan_t1.set_index('filename')['kondisi_akustik']

    baris = []
    for k in kondisi_t1:
        a, p = PREDIKSI[k]
        rinci = pd.DataFrame({
            'filename': a['filename'].values,
            'varian': peta_varian.loc[a['filename'].values].values,
            'akustik': peta_akustik.loc[a['filename'].values].values,
            'intent': a['intent'].astype(str).values == p['intent'].astype(str).values,
        })
        for sl in SLOT:
            rinci[sl] = [samakan(a[sl].values[i]) == samakan(p[sl].values[i])
                         for i in range(len(a))]
        rinci['gabungan'] = rinci[['intent'] + SLOT].all(axis=1)

        for kolom, label in (('varian', 'varian'), ('akustik', 'akustik')):
            for nilai, g in rinci.groupby(kolom):
                baris.append({
                    'kondisi': k,
                    'kelompok': f'{label}: {nilai}',
                    'n': len(g),
                    'intent_%': round(100 * g['intent'].mean(), 1),
                    'produk_%': round(100 * g['produk'].mean(), 1),
                    'jumlah_%': round(100 * g['jumlah'].mean(), 1),
                    'harga_%': round(100 * g['harga'].mean(), 1),
                    'gabungan_%': round(100 * g['gabungan'].mean(), 1),
                })
        baris.append({
            'kondisi': k, 'kelompok': 'SEMUA', 'n': len(rinci),
            'intent_%': round(100 * rinci['intent'].mean(), 1),
            'produk_%': round(100 * rinci['produk'].mean(), 1),
            'jumlah_%': round(100 * rinci['jumlah'].mean(), 1),
            'harga_%': round(100 * rinci['harga'].mean(), 1),
            'gabungan_%': round(100 * rinci['gabungan'].mean(), 1),
        })

    tabel_varian = pd.DataFrame(baris)
    tabel_varian.to_csv(os.path.join(KELUARAN, 'metrik_per_varian_tahap1.csv'), index=False)
    print(tabel_varian.to_string(index=False))

    # Grafik akurasi gabungan per varian ucapan.
    varian_urut = sorted(set(v.split(': ')[1] for v in tabel_varian.kelompok
                             if v.startswith('varian: ')))
    if varian_urut:
        fig, sumbu = plt.subplots(figsize=(8, 4.2))
        lebar = 0.8 / max(len(kondisi_t1), 1)
        posisi_v = np.arange(len(varian_urut))
        for i, k in enumerate(kondisi_t1):
            sub = tabel_varian[tabel_varian.kondisi == k].set_index('kelompok')
            nilai = [sub.loc[f'varian: {v}', 'gabungan_%'] if f'varian: {v}' in sub.index else 0
                     for v in varian_urut]
            bar = sumbu.bar(posisi_v + i * lebar, nilai, lebar,
                            label=label_kondisi.get(k, k).replace(chr(10), ' '))
            sumbu.bar_label(bar, fmt='%.1f', fontsize=7, padding=2)
        sumbu.set_xticks(posisi_v + 0.4 - lebar / 2)
        sumbu.set_xticklabels(varian_urut)
        sumbu.set_ylabel('akurasi gabungan (%)'); sumbu.set_ylim(0, 108)
        sumbu.set_title('Akurasi gabungan NLU per bentuk ucapan (korpus Tahap 1)')
        sumbu.legend(fontsize=8); sumbu.grid(axis='y', alpha=0.3)
        plt.tight_layout()
        plt.savefig(os.path.join(KELUARAN, 'akurasi_per_varian_tahap1.png'), dpi=150)
        plt.show()

**Insight:**

Perhatikan kolom `jumlah_%` dan `harga_%` pada varian `produk`. Di sana acuannya
kosong, sehingga nilai tinggi berarti sistem berhasil **menahan diri** — tidak
mengarang jumlah dari kata bilangan yang sebenarnya bagian nama produk
("enam ratus mililiter"). Itu justru pengujian yang paling langsung bagi aturan
satuan hitung pada rancangan NLU.

Perbandingan baris `akustik: bersih` dan `akustik: teraugmentasi` memisahkan
kesalahan aturan dari kesalahan transkripsi. Pada kondisi teks acuan keduanya
seharusnya nyaris sama, sebab teksnya sama-sama bersih — augmentasi hanya
menyentuh audio. Bila di kondisi ASR keduanya berjauhan, yang menurunkan angka
adalah derau dan kompresi, bukan pemahaman maknanya.

# **MATRIKS KEBINGUNGAN INTENT**

Tahap ini menampilkan matriks kebingungan (*confusion matrix*) klasifikasi
intent untuk tiap kondisi, guna melihat pasangan kelas mana yang tertukar.

In [ ]:
fig, ax = plt.subplots(1, len(kondisi_urut), figsize=(5.4 * len(kondisi_urut), 4.6),
                       squeeze=False)
for i, k in enumerate(kondisi_urut):
    a, p = PREDIKSI[k]
    kelas = sorted(set(a['intent'].astype(str)) | set(p['intent'].astype(str)))
    cm = confusion_matrix(a['intent'].astype(str), p['intent'].astype(str), labels=kelas)
    sumbu = ax[0][i]
    gbr = sumbu.imshow(cm, cmap='Blues')
    sumbu.set_xticks(range(len(kelas))); sumbu.set_xticklabels(kelas, rotation=45, ha='right', fontsize=8)
    sumbu.set_yticks(range(len(kelas))); sumbu.set_yticklabels(kelas, fontsize=8)
    sumbu.set_xlabel('prediksi'); sumbu.set_ylabel('acuan')
    sumbu.set_title(label_kondisi.get(k, k).replace('\n', ' '), fontsize=10)
    batas = cm.max() / 2 if cm.max() else 1
    for r in range(len(kelas)):
        for c in range(len(kelas)):
            if cm[r, c]:
                sumbu.text(c, r, cm[r, c], ha='center', va='center', fontsize=7,
                           color='white' if cm[r, c] > batas else 'black')
    fig.colorbar(gbr, ax=sumbu, fraction=0.046)
plt.tight_layout()
plt.savefig(os.path.join(KELUARAN, 'matriks_kebingungan_intent.png'), dpi=150)
plt.show()

for k in kondisi_urut:
    a, p = PREDIKSI[k]
    # Daftar kelas dihitung ULANG di sini. Memakai variabel `kelas` sisa dari
    # perulangan sebelumnya membuat laporan satu kondisi memakai daftar kelas
    # kondisi lain — barisnya terisi kelas bersupport 0 dan rata-rata makronya
    # berbeda dari tabel ringkas tanpa alasan yang terlihat.
    kelas_k = sorted(set(a['intent'].astype(str)) | set(p['intent'].astype(str)))
    print('=' * 70)
    print(f'Laporan klasifikasi intent — {label_kondisi.get(k, k)}')
    print(f'(kelas yang benar-benar muncul: {len(kelas_k)})')
    print('=' * 70)
    print(classification_report(a['intent'].astype(str), p['intent'].astype(str),
                                labels=kelas_k, zero_division=0, digits=3))

**Insight:**

Pada teks ASR, kesalahan intent paling sering berupa kelas sebenarnya yang
jatuh ke `tanpa_penanda`: kata pembuka rusak melewati ambang kemiripan, sehingga
sistem menganggap ucapan tidak berpenanda. Dampak praktisnya kecil — seluruh
kata intent dalam domain ini, kecuali `beli`, dipetakan ke intent kanonik yang
sama (`catat_penjualan`), dan kolom `akurasi_intent_kanonik` pada tabel ringkas
menunjukkan seberapa banyak kesalahan permukaan itu yang sebenarnya tidak
mengubah tindakan aplikasi.

Yang perlu diawasi adalah kebingungan yang melibatkan `beli`, sebab hanya kelas
itu yang mengubah arah transaksi dari penjualan menjadi pembelian.

# **LOSS**

NLU berbasis aturan tidak memiliki fungsi loss yang dioptimalkan: tidak ada
bobot yang dilatih, dan keluarannya bukan distribusi probabilitas. Karena itu
*cross-entropy* tidak terdefinisi untuknya, dan melaporkannya hanya akan
menciptakan angka yang terlihat ilmiah tanpa dasar.

Yang berlaku untuk pengklasifikasi deterministik adalah dua ukuran berikut:

- **0-1 loss** — proporsi ucapan yang intent-nya salah, yaitu $1 - \text{akurasi}$.
- **Hamming loss** — proporsi slot yang salah dari seluruh slot (4 slot per
  ucapan: intent, produk, jumlah, harga). Lebih halus daripada 0-1 loss karena
  ucapan yang salah pada satu slot tidak dihitung sama dengan yang salah pada
  seluruh slot.

Sebagai pelengkap, bagian ini juga menampilkan **kurva loss pelatihan model
CNN-BiGRU-CTC** bila riwayat pelatihannya tersedia. Loss itu milik komponen
transkripsi (CTC loss), bukan milik NLU — keduanya ditampilkan berdampingan
justru agar tidak tertukar.

In [ ]:
tabel_loss = tabel_ringkas[['kondisi', 'n_ucapan', 'akurasi_intent',
                            'loss_0_1_intent', 'loss_hamming',
                            'akurasi_gabungan']].copy()
for k in ['akurasi_intent', 'loss_0_1_intent', 'loss_hamming', 'akurasi_gabungan']:
    tabel_loss[k] = tabel_loss[k].round(4)
tabel_loss.to_csv(os.path.join(KELUARAN, 'loss_nlu.csv'), index=False)
print('Loss NLU berbasis aturan (bukan cross-entropy — lihat penjelasan di atas):')
print(tabel_loss.to_string(index=False))

# ─── Kurva loss pelatihan model CTC, bila riwayatnya tersedia ───
riwayat = None
for pola in ['../hasil/**/history*.json', '../../ai-service/**/history*.json',
             '/kaggle/input/**/history*.json', '/kaggle/working/**/history*.json',
             './history*.json']:
    hit = sorted(glob.glob(pola, recursive=True))
    if hit:
        with open(hit[0], encoding='utf-8') as f:
            riwayat = json.load(f)
        print(f'\nRiwayat pelatihan ditemukan: {hit[0]}')
        break

if riwayat and 'loss' in riwayat:
    plt.figure(figsize=(7, 4.4))
    plt.plot(riwayat['loss'], label='loss latih')
    if 'val_loss' in riwayat:
        plt.plot(riwayat['val_loss'], label='loss validasi')
    plt.xlabel('epoch'); plt.ylabel('CTC loss')
    plt.title('Kurva loss pelatihan model CNN-BiGRU-CTC')
    plt.legend(); plt.grid(alpha=0.3)
    plt.savefig(os.path.join(KELUARAN, 'kurva_loss_ctc.png'), dpi=150)
    plt.show()
else:
    print('\nRiwayat pelatihan (history*.json) tidak ditemukan, kurva loss CTC dilewati.')
    print('Kurva tersebut dihasilkan oleh notebook pelatihan; simpan '
          'history.history ke JSON di sana agar dapat ditampilkan di sini:')
    print("    json.dump(history.history, open('history.json','w'))")

**Insight:**

Loss NLU dan loss CTC menjawab hal yang berbeda dan tidak boleh dijumlahkan atau
dibandingkan langsung. CTC loss mengukur seberapa baik model menjelaskan audio
sebagai urutan karakter, dan nilainya turun selama pelatihan. 0-1 loss serta
Hamming loss mengukur seberapa sering pemahaman makna keliru, dan nilainya tidak
pernah "dilatih" — ia hanya mencerminkan mutu aturan dan mutu teks masukan.

Hamming loss selalu lebih kecil daripada 1 - akurasi gabungan. Selisih keduanya
memberi tahu bentuk kesalahannya: bila jauh lebih kecil, kesalahan menumpuk pada
satu slot saja per ucapan (biasanya harga), sehingga perbaikan satu komponen
dapat menaikkan akurasi gabungan secara berarti.

# **UJI PENOLAKAN: PRODUK DI LUAR KATALOG**

Seluruh metrik di atas punya satu batas yang harus dinyatakan: **produk
sebenarnya dari setiap ucapan selalu ada di katalog.** Karena itu tugas entitas
produk sesungguhnya adalah memilih satu dari 120 nama, bukan mengekstraksi nama
yang belum tentu dikenal. Itulah sebab F1 produk mendekati sempurna, dan angka
itu tidak menjawab pertanyaan yang pasti muncul di lapangan: *bagaimana kalau
pedagang menyebut produk yang tidak ada di daftar?*

Bagian ini mengujinya tanpa data baru. Sebagian produk **dihapus dari katalog**,
lalu NLU dijalankan pada ucapan yang produknya termasuk yang dihapus. Jawaban
yang benar untuk ucapan itu adalah "tidak dikenali" — bukan produk lain yang
kebetulan mirip. Ucapan yang produknya masih ada di katalog ikut diuji sebagai
pembanding, untuk memastikan penolakan tidak menjadi terlalu galak.

Dua angka yang dihasilkan, dan keduanya penting untuk kasir:

- **Penolakan benar** — ucapan di luar katalog yang berhasil ditolak. Rendah
  berarti transaksi salah produk akan tercatat diam-diam.
- **Penolakan keliru** — ucapan yang produknya ada di katalog tetapi ikut
  ditolak. Tinggi berarti kasir sering disuruh mengulang tanpa alasan.

In [ ]:
FRAKSI_PRODUK_DIHAPUS = 0.2

rng_tolak = np.random.RandomState(SEED)
# Yang diacak adalah INDEKS, bukan daftar Python-nya. np.random.shuffle atas
# list biasa dapat menghasilkan urutan berbeda antar versi numpy, sehingga
# produk yang disembunyikan ikut berbeda antara Kaggle dan komputer lokal —
# dan tabel penolakan di bawah jadi tidak dapat dibandingkan antar-jalan.
urutan = rng_tolak.permutation(len(PRODUK_URUT))
n_hapus = max(1, int(len(PRODUK_URUT) * FRAKSI_PRODUK_DIHAPUS))
produk_dihapus = {PRODUK_URUT[i] for i in urutan[:n_hapus]}
katalog_kurang = [p for p in PRODUK_URUT if p not in produk_dihapus]

print(f'Katalog dikurangi: {len(PRODUK_URUT)} -> {len(katalog_kurang)} produk '
      f'({n_hapus} nama disembunyikan)')

nlu_kurang = NLUAturan(katalog_kurang,
                       {k: v for k, v in HARGA_SATUAN.items() if k not in produk_dihapus})

baris = []
for nama_kondisi, (a, p) in PREDIKSI.items():
    di_luar = a['produk'].isin(produk_dihapus).values
    if di_luar.sum() == 0:
        continue
    teks = p['teks_masukan'].astype(str).values
    hasil = [nlu_kurang.parse(t) for t in teks]
    ditolak = np.array([h['produk'] is None for h in hasil])

    # Untuk ucapan di dalam katalog, periksa juga apakah produknya masih benar.
    benar_di_dalam = np.array([
        h['produk'] == g for h, g in zip(hasil, a['produk'].values)])

    baris.append({
        'kondisi': nama_kondisi,
        'n_di_luar_katalog': int(di_luar.sum()),
        'penolakan_benar_%': round(100 * ditolak[di_luar].mean(), 2),
        'n_di_dalam_katalog': int((~di_luar).sum()),
        'penolakan_keliru_%': round(100 * ditolak[~di_luar].mean(), 2),
        'produk_benar_di_dalam_%': round(100 * benar_di_dalam[~di_luar].mean(), 2),
    })

tabel_tolak = pd.DataFrame(baris)
tabel_tolak.to_csv(os.path.join(KELUARAN, 'uji_penolakan_luar_katalog.csv'), index=False)
print()
print(tabel_tolak.to_string(index=False))

# Bagaimana ambang mempengaruhi keseimbangan keduanya.
print('\nPengaruh ambang pada kondisi paling menantang:')
kondisi_sulit = kondisi_urut[-1]
a, p = PREDIKSI[kondisi_sulit]
di_luar = a['produk'].isin(produk_dihapus).values
hasil = [nlu_kurang.cocokkan_produk(nlu_kurang.parse(t)['span_produk'], gerbang=False)
         for t in p['teks_masukan'].astype(str)]
skor = np.array([s for _, s in hasil])
print(f'  ({kondisi_sulit}, {int(di_luar.sum())} ucapan di luar katalog)')
for t in [0.30, 0.40, 0.45, 0.50, 0.60, 0.70]:
    tolak = skor < t
    print(f'   ambang {t:.2f}: penolakan benar {100*tolak[di_luar].mean():5.1f}% | '
          f'penolakan keliru {100*tolak[~di_luar].mean():5.1f}%')

**Insight:**

Angka penolakan inilah yang menentukan apakah sistem aman dipakai di warung yang
katalognya belum lengkap — dan ia tidak terlihat sama sekali pada F1 produk.
Sebuah sistem bisa saja mencapai F1 99% pada produk yang dikenal sambil memetakan
setiap ucapan asing ke produk termirip, dan kesalahan seperti itu justru yang
paling mahal: transaksi tetap tercatat, hanya dengan barang yang salah.

Bacalah kedua kolom bersama-sama. Penolakan benar yang tinggi tetapi disertai
penolakan keliru yang tinggi pula bukan keberhasilan — itu hanya ambang yang
terlalu ketat, dan biayanya ditanggung kasir dalam bentuk permintaan mengulang.
Tabel ambang di bawahnya memperlihatkan pertukaran itu secara langsung, sehingga
nilai `AMBANG_PRODUK` dapat dipilih berdasarkan angka, bukan selera.

# **ANALISIS SENSITIVITAS AMBANG PRODUK**

Tahap ini memeriksa pengaruh `AMBANG_PRODUK` terhadap hasil. Ambang yang terlalu
rendah membuat sistem menebak produk pada ucapan yang tidak jelas; terlalu
tinggi membuatnya menolak ucapan yang sebenarnya benar.

Karena kandidat terbaik dan skornya sudah disimpan saat pengurai berjalan,
pengaruh tiap ambang dapat dihitung tanpa mengurai ulang seluruh data.

In [ ]:
ambang_uji = np.round(np.arange(0.20, 0.86, 0.05), 2)
baris = []
for k in kondisi_urut:
    a, p = PREDIKSI[k]
    for t in ambang_uji:
        produk_t = [kand if skor >= t else None
                    for kand, skor in zip(p['produk_kandidat'], p['skor_produk'])]
        m = metrik_slot(a['produk'].values, produk_t)
        # "Salah terima": sistem mengisi produk, tetapi produknya keliru.
        salah_terima = sum(1 for g, h in zip(a['produk'].values, produk_t)
                           if h is not None and samakan(g) != samakan(h))
        baris.append({'kondisi': k, 'ambang': t, 'f1_produk': m['f1'],
                      'presisi': m['presisi'], 'recall': m['recall'],
                      'salah_terima_%': 100 * salah_terima / max(len(produk_t), 1)})
sensitivitas = pd.DataFrame(baris)
sensitivitas.to_csv(os.path.join(KELUARAN, 'sensitivitas_ambang.csv'), index=False)

fig, ax = plt.subplots(1, 2, figsize=(12, 4.2))
for k in kondisi_urut:
    sub = sensitivitas[sensitivitas.kondisi == k]
    ax[0].plot(sub.ambang, 100 * sub.f1_produk, marker='o', ms=3,
               label=label_kondisi.get(k, k).replace('\n', ' '))
    ax[1].plot(sub.ambang, sub['salah_terima_%'], marker='o', ms=3,
               label=label_kondisi.get(k, k).replace('\n', ' '))
for sumbu, judul, ylab in [(ax[0], 'F1 entitas produk vs ambang', 'F1 produk (%)'),
                           (ax[1], 'Salah terima vs ambang', 'ucapan salah terima (%)')]:
    sumbu.axvline(AMBANG_PRODUK, color='black', ls='--', lw=1,
                  label=f'ambang terpakai {AMBANG_PRODUK}')
    sumbu.set_xlabel('ambang kemiripan produk'); sumbu.set_ylabel(ylab)
    sumbu.set_title(judul); sumbu.legend(fontsize=7); sumbu.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(KELUARAN, 'sensitivitas_ambang.png'), dpi=150)
plt.show()

terbaik = (sensitivitas.groupby('ambang').f1_produk.mean().idxmax())
print(f'Ambang dengan F1 produk rata-rata tertinggi lintas kondisi: {terbaik}')
print(f'Ambang yang dipakai pada evaluasi ini                     : {AMBANG_PRODUK}')
print('\nUbah AMBANG_PRODUK di sel konfigurasi lalu jalankan ulang notebook '
      'bila ingin memakai nilai lain — seluruh tabel akan ikut menyesuaikan.')

**Insight:**

Kurva F1 biasanya landai di sekitar nilai terbaiknya, sedangkan kurva salah
terima menanjak tajam begitu ambang diturunkan. Bentuk ini mendukung pemilihan
ambang yang sedikit lebih tinggi daripada titik F1 tertinggi: F1 hanya berkurang
sedikit, sementara kesalahan yang paling merugikan pencatatan transaksi —
produk salah yang terlanjur tercatat — berkurang jauh lebih banyak.

# **ANALISIS KESALAHAN**

Tahap ini memeriksa kesalahan yang sesungguhnya terjadi dan memisahkan
penyebabnya: kesalahan yang berasal dari aturan NLU sendiri, dan kesalahan yang
berasal dari teks transkripsi yang sudah rusak sejak awal.

Pemisahan dilakukan pada **irisan ucapan yang sama**: ucapan yang benar pada
kondisi teks acuan tetapi salah pada kondisi ASR pasti gagal karena
transkripsinya, bukan karena aturannya.

In [ ]:
def daftar_benar(nama):
    a, p = PREDIKSI[nama]
    benar = {}
    for i in range(len(a)):
        cocok = (str(a['intent'].values[i]) == str(p['intent'].values[i])
                 and all(samakan(a[s].values[i]) == samakan(p[s].values[i]) for s in SLOT))
        benar[a['filename'].values[i]] = cocok
    return benar


BENAR = {k: daftar_benar(k) for k in kondisi_urut}

for k in kondisi_urut:
    if not k.startswith('asr_'):
        continue
    kunci_acuan = SUMBER_ACUAN.get(k, 'teks_acuan')
    if kunci_acuan not in BENAR:
        print(f'[LEWATI] {k}: kondisi teks acuan pasangannya tidak dievaluasi.')
        continue
    a, p = PREDIKSI[k]
    benar_acuan = BENAR[kunci_acuan]
    benar_asr = BENAR[k]
    irisan = [f for f in benar_asr if f in benar_acuan]
    rusak_karena_asr = sum(1 for f in irisan if benar_acuan[f] and not benar_asr[f])
    gagal_dua_duanya = sum(1 for f in irisan if not benar_acuan[f] and not benar_asr[f])
    benar_dua_duanya = sum(1 for f in irisan if benar_acuan[f] and benar_asr[f])

    print('=' * 74)
    print(f'{label_kondisi.get(k, k).replace(chr(10), " ")} — irisan {len(irisan):,} ucapan')
    print('=' * 74)
    print(f'  benar pada kedua kondisi          : {benar_dua_duanya:,} '
          f'({100*benar_dua_duanya/max(len(irisan),1):.1f}%)')
    print(f'  rusak KARENA transkripsi          : {rusak_karena_asr:,} '
          f'({100*rusak_karena_asr/max(len(irisan),1):.1f}%)')
    print(f'  gagal walau teksnya benar (aturan): {gagal_dua_duanya:,} '
          f'({100*gagal_dua_duanya/max(len(irisan),1):.1f}%)')

    # Contoh kesalahan yang disebabkan transkripsi.
    contoh = 0
    for i in range(len(a)):
        f = a['filename'].values[i]
        if benar_acuan.get(f) and not benar_asr.get(f):
            print(f'\n  [{f}]')
            print(f'    teks ASR : {p["teks_masukan"].values[i]}')
            print(f'    acuan    : intent={a["intent"].values[i]} | produk={a["produk"].values[i]} '
                  f'| jumlah={a["jumlah"].values[i]} | harga={a["harga"].values[i]}')
            print(f'    prediksi : intent={p["intent"].values[i]} | produk={p["produk"].values[i]} '
                  f'| jumlah={p["jumlah"].values[i]} | harga={p["harga"].values[i]}')
            contoh += 1
            if contoh >= 4:
                break
    print()

# Slot mana yang paling sering salah, per kondisi.
print('Frekuensi kesalahan per slot:')
baris = []
for k in kondisi_urut:
    a, p = PREDIKSI[k]
    rec = {'kondisi': k}
    rec['intent'] = float(np.mean(a['intent'].astype(str).values != p['intent'].astype(str).values))
    for s in SLOT:
        rec[s] = float(np.mean([samakan(a[s].values[i]) != samakan(p[s].values[i])
                                for i in range(len(a))]))
    baris.append(rec)
kesalahan_slot = pd.DataFrame(baris).set_index('kondisi')
kesalahan_slot = (100 * kesalahan_slot).round(2)
kesalahan_slot.to_csv(os.path.join(KELUARAN, 'kesalahan_per_slot.csv'))
print(kesalahan_slot.to_string())

**Insight:**

Pembagian ini menentukan ke mana usaha perbaikan diarahkan, dan jawabannya
hampir selalu berbeda dari dugaan awal.

Bila sebagian besar kesalahan masuk kategori "rusak karena transkripsi",
menyempurnakan aturan NLU tidak akan menolong — yang perlu diperbaiki adalah
model ASR-nya, atau jalur penyangga di sisi aplikasi seperti penskoran ulang
terhadap katalog produk. Sebaliknya, ucapan yang gagal walau teksnya sudah benar
menunjuk langsung ke celah pada aturan, dan biasanya jumlahnya kecil serta dapat
ditutup dengan menambah satu pola.

# **EKSPOR HASIL**

In [ ]:
for nama, (a, p) in PREDIKSI.items():
    gabung = a[['filename', 'teks', 'intent', 'produk', 'jumlah', 'harga']].copy()
    gabung.columns = ['filename', 'teks_acuan', 'intent_acuan', 'produk_acuan',
                      'jumlah_acuan', 'harga_acuan']
    gabung['teks_masukan'] = p['teks_masukan'].values
    gabung['intent_prediksi'] = p['intent'].values
    gabung['produk_prediksi'] = p['produk'].values
    gabung['jumlah_prediksi'] = p['jumlah'].values
    gabung['harga_prediksi'] = p['harga'].values
    gabung['skor_produk'] = p['skor_produk'].values
    gabung.to_csv(os.path.join(KELUARAN, f'prediksi_nlu_{nama}.csv'), index=False)

ringkasan_json = {
    'transkrip_acuan': TRANSCRIPTS,
    'katalog': KATALOG,
    'ambang_produk': AMBANG_PRODUK,
    'jumlah_produk_katalog': len(PRODUK_URUT),
    'kondisi': {k: int(len(v)) for k, v in KONDISI.items()},
    'metrik': tabel_ringkas.to_dict(orient='records'),
    'waktu_evaluasi': time.strftime('%Y-%m-%d %H:%M:%S'),
}
with open(os.path.join(KELUARAN, 'ringkasan_nlu.json'), 'w', encoding='utf-8') as f:
    json.dump(ringkasan_json, f, indent=2, ensure_ascii=False)

with open(os.path.join(KELUARAN, 'metrik_nlu.md'), 'w', encoding='utf-8') as f:
    f.write(tampil.to_markdown(index=False))

print('Berkas keluaran di', os.path.abspath(KELUARAN) + ':')
for nama in sorted(os.listdir(KELUARAN)):
    ukuran = os.path.getsize(os.path.join(KELUARAN, nama))
    print(f'  {nama:36s} {ukuran/1024:8.1f} KB')

# **KESIMPULAN**

In [ ]:
print('=' * 78)
print('RINGKASAN EVALUASI NLU BERBASIS ATURAN')
print('=' * 78)
for _, r in tabel_ringkas.iterrows():
    print(f'\n{label_kondisi.get(r.kondisi, r.kondisi).replace(chr(10), " ")}  '
          f'({int(r.n_ucapan):,} ucapan)')
    print(f'  akurasi intent        : {100*r.akurasi_intent:.2f}%  '
          f'(kanonik: {100*r.akurasi_intent_kanonik:.2f}%)')
    print(f'  F1 macro intent       : {100*r.f1_macro_intent:.2f}%')
    print(f'  F1 weighted intent    : {100*r.f1_weighted_intent:.2f}%')
    print(f'  F1 mikro entitas      : {100*r.f1_mikro_entitas:.2f}%  '
          f'(produk {100*r.f1_produk:.1f} | jumlah {100*r.f1_jumlah:.1f} | harga {100*r.f1_harga:.1f})')
    print(f'  akurasi gabungan      : {100*r.akurasi_gabungan:.2f}%')
    print(f'  0-1 loss / Hamming    : {r.loss_0_1_intent:.4f} / {r.loss_hamming:.4f}')

if 'asr_speaker_independent' in tabel_ringkas.kondisi.values:
    acu = tabel_ringkas[tabel_ringkas.kondisi == 'teks_acuan'].iloc[0]
    asr = tabel_ringkas[tabel_ringkas.kondisi == 'asr_speaker_independent'].iloc[0]
    print(f'\nBiaya kesalahan transkripsi (teks acuan -> ASR speaker-independent):')
    print(f'  akurasi intent   : {100*(asr.akurasi_intent - acu.akurasi_intent):+.2f} poin persen')
    print(f'  F1 mikro entitas : {100*(asr.f1_mikro_entitas - acu.f1_mikro_entitas):+.2f} poin persen')
    print(f'  akurasi gabungan : {100*(asr.akurasi_gabungan - acu.akurasi_gabungan):+.2f} poin persen')
else:
    print('\nKondisi ASR speaker-independent belum tersedia — jalankan notebook 07 '
          'dengan checkpoint Skema B untuk melengkapi perbandingan ini.')

if 'asr_tahap1_val' in tabel_ringkas.kondisi.values:
    t1 = tabel_ringkas[tabel_ringkas.kondisi == 'asr_tahap1_val'].iloc[0]
    print('\nKorpus Tahap 1 (model v12 — yang benar-benar dilayankan aplikasi):')
    print(f'  akurasi gabungan : {100*t1.akurasi_gabungan:.2f}% atas {int(t1.n_ucapan):,} klip')
    print('  Catatan kesahihan: klip validasi Tahap 1 dipisah per REKAMAN sumber,')
    print('  bukan per penutur. Angka ini speaker-dependent pada rekaman baru, dan')
    print('  tidak dapat menggantikan kondisi speaker-independent di atas.')

**Insight akhir:**

Hasil pada kondisi **teks acuan** menjawab RM-4: aturan yang dirancang mampu
mengubah kalimat transaksi menjadi struktur `intent + produk + jumlah + harga`.
Angkanya tinggi, dan memang seharusnya begitu — pada teks yang bersih, domain
sesempit ini memang dapat ditangani aturan tanpa model terlatih.

Hasil pada kondisi **ASR speaker-independent** menjawab RM-5 untuk kondisi yang
sebenarnya dihadapi aplikasi. Angka itulah yang pantas dikutip sebagai
performansi sistem, sebab pedagang tidak mengetikkan teks bersih — mereka
berbicara, dan teksnya datang dari model yang belum pernah mendengar suara
mereka.

Dua batasan yang perlu dinyatakan bersama angka mana pun di notebook ini.
Pertama, evaluasi dilakukan pada kalimat yang berpola sama dengan data latih;
ucapan di luar pola (kalimat majemuk, dua produk sekaligus, tawar-menawar) tidak
terwakili. Kedua, label acuan dibangun dari struktur dataset, sehingga sah
selama dataset itu memang terbentuk seperti yang diasumsikan — dan itulah yang
diperiksa oleh pemeriksaan silang harga di awal notebook.